# CEval Demo — Adult Income Dataset

**CEval** is a Python package for evaluating counterfactual explanations across 14 quality metrics.

> **Paper:** Bayrak, B., & Bach, K. (2024). *Evaluation of Instance-Based Explanations: An In-Depth Analysis of Counterfactual Evaluation Metrics, Challenges, and the CEval Toolkit.*  
> IEEE Access. [doi:10.1109/ACCESS.2024.3410540](https://ieeexplore.ieee.org/document/10550910)

This notebook demonstrates how to:
1. Load and prepare the **Adult Income** dataset
2. Train a **Random Forest** classifier
3. Generate counterfactuals using **DiCE**
4. Evaluate them with **CEval** in both *1-to-1* and *1-to-N* mode
5. Visualise and interpret the results

---


## 0 · Installation

In [ ]:
# Install required packages (run once)
# !pip install CEval dice-ml scikit-learn pandas numpy matplotlib seaborn

## 1 · Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import dice_ml
from dice_ml import Dice

from ceval import CEval   # pip install CEval

pd.set_option("display.float_format", "{:.3f}".format)

## 2 · Dataset

We use the **Adult Income** dataset (Dua & Graff, 2019) which is a classic benchmark for counterfactual XAI research.
The task is to predict whether a person earns **more than $50K/year**.

Features used: `age`, `education_num`, `hours_per_week`, `capital_gain`, `capital_loss`


In [ ]:
LABEL        = "income"
FEATURE_COLS = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]

url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
)
col_names = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", LABEL,
]

try:
    df_raw = pd.read_csv(url, names=col_names, sep=",\\s*",
                         engine="python", na_values="?").dropna()
    df_raw[LABEL] = df_raw[LABEL].str.strip().map({"<=50K": 0, ">50K": 1}).astype(int)
    df = df_raw[FEATURE_COLS + [LABEL]].sample(n=2000, random_state=42).reset_index(drop=True)
    print("Loaded UCI Adult Income dataset")
except Exception:
    rng = np.random.default_rng(42)
    n   = 1000
    age          = rng.integers(18, 70, n)
    education_num= rng.integers(1, 16, n)
    hours_per_week=rng.integers(20, 60, n)
    capital_gain = (rng.random(n) > 0.85).astype(int) * rng.integers(1000, 10000, n)
    capital_loss = (rng.random(n) > 0.92).astype(int) * rng.integers(100, 2000, n)
    score  = 0.05 * education_num + 0.01 * hours_per_week + 0.0001 * capital_gain
    income = (score > rng.uniform(0.4, 1.2, n)).astype(int)
    df = pd.DataFrame(dict(age=age, education_num=education_num,
                           hours_per_week=hours_per_week, capital_gain=capital_gain,
                           capital_loss=capital_loss, income=income))
    print("Generated synthetic data (offline fallback)")

print(f"   Shape: {df.shape}")
df.head()

In [ ]:
# Class balance
df[LABEL].value_counts().rename({0: "≤50K", 1: ">50K"}).plot(
    kind="bar", color=["#4C9BE8","#E87A4C"], edgecolor="white", figsize=(4,3)
)
plt.title("Income class distribution"); plt.ylabel("Count")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3 · Train / Test Split & Classifier

In [ ]:
X = df[FEATURE_COLS]
y = df[LABEL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

train_df = X_train.copy(); train_df[LABEL] = y_train.values
test_df  = X_test.copy();  test_df[LABEL]  = y_test.values

# Sample a small set to explain (keeps runtime short)
N_EXPLAIN   = 10
test_sample = test_df.sample(n=N_EXPLAIN, random_state=7).reset_index(drop=True)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
acc = accuracy_score(y_test, clf.predict(X_test))
print(f"RandomForest accuracy: {acc:.2%}")

## 4 · Generate Counterfactuals with DiCE

[DiCE](https://github.com/interpretml/DiCE) (Diverse Counterfactual Explanations) generates counterfactuals
that are both valid and diverse.  We run it in two modes:

- **1-to-1**: one counterfactual per instance  
- **1-to-N**: three counterfactuals per instance


In [ ]:
dice_data  = dice_ml.Data(dataframe=train_df, continuous_features=FEATURE_COLS,
                          outcome_name=LABEL)
dice_model = dice_ml.Model(model=clf, backend="sklearn", model_type="classifier")
explainer  = Dice(dice_data, dice_model, method="random")
print("DiCE explainer initialised")

In [ ]:
def run_dice(n_samples, n_cfs, mode):
    rows = []
    for i in range(n_samples):
        row = test_sample.drop(columns=[LABEL]).iloc[i:i+1]
        res = explainer.generate_counterfactuals(row, total_CFs=n_cfs,
                                                  desired_class="opposite",
                                                  verbose=False)
        cf_df = res.cf_examples_list[0].final_cfs_df
        if cf_df is None or len(cf_df) == 0:
            for _ in range(n_cfs):
                fb = test_sample.iloc[i].copy()
                fb[LABEL] = 1 - fb[LABEL]
                if mode == "1toN": fb["instance"] = i
                rows.append(fb)
        else:
            for _, r in cf_df.iterrows():
                r = r.copy()
                if mode == "1toN": r["instance"] = i
                rows.append(r)
    out = pd.DataFrame(rows).reset_index(drop=True)
    out[LABEL] = out[LABEL].astype(int)
    return out

cfs_1to1 = run_dice(N_EXPLAIN, 1, "1to1")
print(f"1-to-1: {len(cfs_1to1)} counterfactuals")

cfs_1toN = run_dice(N_EXPLAIN, 3, "1toN")
print(f"1-to-N: {len(cfs_1toN)} counterfactuals ({len(cfs_1toN)//N_EXPLAIN} per instance)")

In [ ]:
# Preview a counterfactual alongside its original instance
print("Original instance:")
display(test_sample.iloc[[0]])
print("\nCounterfactual:")
display(cfs_1to1.iloc[[0]])

## 5 · Evaluate with CEval

We set `age` as a constraint (immutable feature, age cannot be reduced).


In [ ]:
evaluator = CEval(
    samples     = test_sample,
    label       = LABEL,
    data        = train_df,
    model       = clf,
    k_nn        = 5,
    constraints = ["age"],
)

evaluator.add_explainer("DiCE 1-to-1", cfs_1to1, "generated-cf", mode="1to1")
evaluator.add_explainer("DiCE 1-to-N", cfs_1toN, "generated-cf", mode="1toN")

print("Evaluation complete")

In [ ]:
# Full results table (transposed for readability)
evaluator.comparison_table.T

## 6 · Visualise Results

In [ ]:
# Filter to numeric metrics only (drop '-')
numeric_results = (
    evaluator.comparison_table
    .replace("-", float("nan"))
    .apply(pd.to_numeric, errors="coerce")
    .dropna(axis=1, how="all")
)

fig, ax = plt.subplots(figsize=(12, 3.5))
numeric_results.T.plot(kind="bar", ax=ax, color=["#4C9BE8","#E87A4C"],
                        edgecolor="white", width=0.7)
ax.set_title("CEval metric comparison — DiCE on Adult Income", fontsize=13, pad=12)
ax.set_xlabel(""); ax.set_ylabel("Score")
ax.legend(frameon=True)
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
# Highlight: metrics where 1-to-N outperforms 1-to-1
# Lower is better for proximity / sparsity / redundancy / constraint_violation
# Higher is better for validity / diversity / yNN

better_lower = {"proximity", "proximity_gower", "sparsity",
                "redundancy", "constraint_violation", "feasibility",
                "kNLN_dist", "relative_dist", "plausibility"}

rows = []
for m in numeric_results.columns:
    v1  = numeric_results.loc["DiCE 1-to-1", m]
    vN  = numeric_results.loc["DiCE 1-to-N", m]
    if pd.isna(v1) or pd.isna(vN):
        continue
    better = "1-to-N ✓" if (m in better_lower and vN < v1) or (m not in better_lower and vN > v1) else "1-to-1 ✓"
    rows.append({"metric": m, "DiCE 1-to-1": v1, "DiCE 1-to-N": vN, "better": better})

summary = pd.DataFrame(rows).set_index("metric")
summary.style.apply(
    lambda col: ["background-color: #d4edda" if v.endswith("✓") else "" for v in col],
    subset=["better"]
)

## 7 · Metric Interpretation Guide

| Metric | Better when | Interpretation |
|---|---|---|
| `validity` | **Higher** | More CFs actually flip the model's prediction |
| `proximity` / `proximity_gower` | **Lower** | CFs are closer to the original instance |
| `sparsity` | **Lower** | Fewer features need to change |
| `count` | depends | More CFs give users more options |
| `diversity` / `diversity_lcc` | **Higher** | CFs cover a wider range of options |
| `yNN` | **Higher** | CF's neighbourhood is consistent with its class |
| `feasibility` | **Lower** | CFs lie close to real training points |
| `kNLN_dist` | **Lower** | CFs are close to like-labelled real instances |
| `relative_dist` | **Lower** | CF distance is small relative to NUN distance |
| `redundancy` | **Lower** | Fewer unnecessary feature changes |
| `plausibility` | **Lower** | CF sits close to the decision boundary |
| `constraint_violation` | **Lower** | Fewer immutable features changed |

---

## 8 · Next Steps

- Swap DiCE for another explainer (PertCF, NICE, WACHTER, etc.) and compare  
- Try the **1-to-N** mode to evaluate explainer diversity more richly  
- Add more constraints to reflect domain knowledge  
- See the [GitHub repository](https://github.com/b-bayrak/CEval) for more examples
